In [1]:
# Import relevant libraries.
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from datetime import datetime
import re

import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('words')
nltk.download('omw-1.4')
from nltk.corpus import stopwords
from nltk.corpus import words
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to C:\Users\Jai
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Jai
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package words to C:\Users\Jai
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Jai
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
# Load dataset. Change directory as required.
df = pd.read_csv('japan_speeches.csv')

In [3]:
df.head()

,reference,country,date,title,author,is_gov,text
0,r970204a_BOJ,japan,04/02/1997,Recent Monetary and Economic Conditions in Japan,matsushita,0,I truly appreciate this opportunity to address...
1,r970228a_BOJ,japan,28/02/1997,Payment and Settlement Systems: The Current Is...,matsushita,0,This article is excerpted and translated from ...
2,r970414a_BOJ,japan,14/04/1997,Recent Monetary and Economic Conditions in Jap...,matsushita,0,This article is excerpted and translated from ...
3,r970627a_BOJ,japan,27/06/1997,A New Framework of Monetary Policy under the N...,matsushita,0,This article is excerpted and translated from ...
4,r970829a_BOJ,japan,29/08/1997,Foreword by the Governor,governor,0,It is a pleasure to be able to bring to you th...


In [4]:
df.country.value_counts()

japan    753
Name: country, dtype: int64

In [5]:
# Add a column to calculate the string length per speech.
df['len'] = df['text'].str.len()
df

,reference,country,date,title,author,is_gov,text,len
0,r970204a_BOJ,japan,04/02/1997,Recent Monetary and Economic Conditions in Japan,matsushita,0,I truly appreciate this opportunity to address...,32759
1,r970228a_BOJ,japan,28/02/1997,Payment and Settlement Systems: The Current Is...,matsushita,0,This article is excerpted and translated from ...,32759
2,r970414a_BOJ,japan,14/04/1997,Recent Monetary and Economic Conditions in Jap...,matsushita,0,This article is excerpted and translated from ...,32759
3,r970627a_BOJ,japan,27/06/1997,A New Framework of Monetary Policy under the N...,matsushita,0,This article is excerpted and translated from ...,32759
4,r970829a_BOJ,japan,29/08/1997,Foreword by the Governor,governor,0,It is a pleasure to be able to bring to you th...,4629
...,...,...,...,...,...,...,...,...
748,r220728a_BOJ,japan,28/07/2022,Japan's Economy and Monetary Policy,amamiya,0,It is my pleasure to have the opportunity toda...,22639
749,r220825a_BOJ,japan,25/08/2022,"Economic Activity, Prices, and Monetary Policy...",nakamura,0,I will begin my speech by talking about recent...,22720
750,r220831a_BOJ,japan,31/08/2022,"Economic Activity, Prices, and Monetary Policy...",nakagawa,0,I would like to begin my speech by talking abo...,17495
751,r220926a_BOJ,japan,26/09/2022,Japan's Economy and Monetary Policy,kuroda,1,It is my great pleasure to have the opportunit...,16568


In [6]:
# Text cleaning (Convert to lower case and remove punctuation)
df['text'] = df['text'].str.lower().str.replace('[^\w\s]', '', regex=True)

In [7]:
# VADER sentiment (Calculate Sentiment intensity analysis using Vadar sentiment)
sia = SentimentIntensityAnalyzer()
df[['neg', 'neu', 'pos', 'compound']] = df['text'].apply(lambda x: pd.Series(sia.polarity_scores(x)))

In [8]:
# TextBlob sentiment (Calculate polarity and subjectivity using TextBlob)
df[['polarity','subjectivity']] = df['text'].apply(lambda x: pd.Series(TextBlob(x).sentiment))

In [9]:
# Load Loughran–McDonald Dictionary
lm_dict = pd.read_csv("LM_dictionary.csv")  
print("LM Columns:", lm_dict.columns)  # check columns

LM Columns: Index(['Word', 'Negative', 'Positive', 'Uncertainty', 'Litigious', 'Strong',
       'Weak', 'Constraining'],
      dtype='object')


In [10]:
# Create a mapping: Word -> list of categories
lm_dict_map = {}
for _, row in lm_dict.iterrows():
    word = row['Word'].upper()
    categories = [col for col in lm_dict.columns[1:] if row[col] > 0]  # skip 'Word' column
    lm_dict_map[word] = categories

In [11]:
# Function to compute LM sentiment
def lm_sentiment(text, lm_dict_map):
    words = re.findall(r'\b\w+\b', text.upper())
    pos = sum(1 for w in words if 'Positive' in lm_dict_map.get(w, []))
    neg = sum(1 for w in words if 'Negative' in lm_dict_map.get(w, []))
    total = pos + neg
    return 0 if total == 0 else (pos - neg)/total

In [12]:
#import re
# Apply LM sentiment
df['lm_score'] = df['text'].apply(lambda x: lm_sentiment(x, lm_dict_map))

In [13]:
# Combined Score (simple average)
df['combined_score'] = (df['compound'] + df['polarity'] + df['lm_score']) / 3

In [14]:
# LM Label Thresholds
def lm_label(score, pos_thresh=0.05, neg_thresh=-0.05):
    """
    Assigns a sentiment label based on LM score thresholds.
    
    Parameters:
        score: LM sentiment score ([-1,1])
        pos_thresh: threshold above which text is Positive
        neg_thresh: threshold below which text is Negative
        
    Returns:
        'Positive', 'Negative', or 'Neutral'
    """
    if score > pos_thresh:
        return "Positive"
    elif score < neg_thresh:
        return "Negative"
    else:
        return "Neutral"
    
df['lm_label'] = df['lm_score'].apply(lambda x: lm_label(x))    

# Export Selected Columns
columns_to_export = [
    'reference','date','text', 
    'neg', 'neu', 'pos', 'compound', 
    'polarity', 'subjectivity', 
    'lm_score', 
    'combined_score', 
    'lm_label'
]

In [15]:
# Export CSV
df.to_csv("Japan_sentiment_labeled.csv", columns=columns_to_export, index=False, encoding='utf-8')

In [16]:
# Export Excel
df.to_excel("Japan_sentiment_labeled.xlsx", columns=columns_to_export, index=False, sheet_name="Euro_Sentiment")